# 05. Avaliar atribuição — mãe + endereço

Depois do [`04_atribuir_mae_endereco.ipynb`](04_atribuir_mae_endereco.ipynb)
(modelo `p ≥ 0,90` + resgate R1/R2/R3).

Avalia notas, qualidade, quem ficou de fora, amostra por faixa/regra e
números com lista ouro.

Entradas:
- `splink_predictions_mae_endereco.parquet` ([`02d`](02d_aplicar_splink_mae_endereco.ipynb))
- `splink_atribuicao_mae_endereco.parquet` ([`04`](04_atribuir_mae_endereco.ipynb))


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    LISTA_OURO_ARQUIVO,
    MAX_CENSOS_POR_CPF,
    SPLINK_ATRIBUICAO_MAE_ENDERECO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    TABELA_CENSO_LIMPA,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    cpf_norm_sql,
    drop_splink_temp_tables,
    get_connection,
    lista_ouro_colunas,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

PISO_MODELO = 0.90
PISO_RESGATE = 0.50
PISO_BAIXO = 0.25
TETO = MAX_CENSOS_POR_CPF

print_paths()
require_input(
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    label='SPLINK_PREDICTIONS_MAE_ENDERECO (rode o 02d antes)',
)
require_input(
    SPLINK_ATRIBUICAO_MAE_ENDERECO,
    label='SPLINK_ATRIBUICAO_MAE_ENDERECO (rode o 04_atribuir_mae_endereco antes)',
)
require_input(LISTA_OURO_ARQUIVO, label='LISTA_OURO')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA, TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print(f'Faixas 04: modelo>={PISO_MODELO} | resgate>={PISO_RESGATE} | baixo>={PISO_BAIXO} | teto={TETO}')


In [ ]:
_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')

con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS_MAE_ENDERECO}')
''')

# Nome distinto da TABLE atribuicao do 04 no mesmo DuckDB.
for _nome in ('atribuicao_aval', 'atribuicao_2', 'atribuicao'):
    _t = con.execute(f'''
    SELECT table_type
    FROM information_schema.tables
    WHERE table_schema = 'main' AND table_name = '{_nome}'
    ''').fetchone()
    if _t:
        _k = 'VIEW' if _t[0].upper() == 'VIEW' else 'TABLE'
        con.execute(f'DROP {_k} IF EXISTS {_nome}')

cols_atr = [
    r[0]
    for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{SPLINK_ATRIBUICAO_MAE_ENDERECO}') LIMIT 0"
    ).fetchall()
]
tem_faixa = 'faixa' in cols_atr
tem_regra = 'regra' in cols_atr
col_faixa = 'faixa' if tem_faixa else 'CAST(NULL AS VARCHAR) AS faixa'
col_regra = 'regra' if tem_regra else 'CAST(NULL AS VARCHAR) AS regra'

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_aval AS
SELECT
    unique_id_censo,
    unique_id_cpf,
    match_probability,
    {col_faixa},
    {col_regra}
FROM read_parquet('{SPLINK_ATRIBUICAO_MAE_ENDERECO}')
''')

n_app = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''').fetchone()[0]
n_atrib = con.execute('SELECT COUNT(*) FROM atribuicao_aval').fetchone()[0]
n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]

print(f'Aplicação: {n_app:,}')
print(f'Atribuídos: {n_atrib:,} ({100.0 * n_atrib / n_app:.1f}%)')
print(f'Sem CPF: {n_app - n_atrib:,} ({100.0 * (n_app - n_atrib) / n_app:.1f}%)')
print(f'Pares no parquet: {n_pred:,}')
print('Colunas faixa/regra:', tem_faixa, tem_regra)


## Notas e regra dos atribuídos

Distribuição de `match_probability`, contagem por `faixa`/`regra` do 04, e
Censos por CPF.


In [ ]:
display(con.execute('''
SELECT
    CASE
        WHEN match_probability >= 0.99 THEN 'p_ge_099'
        WHEN match_probability >= 0.90 THEN 'p_090_099'
        WHEN match_probability >= 0.75 THEN 'p_075_090'
        WHEN match_probability >= 0.50 THEN 'p_050_075'
        WHEN match_probability >= 0.25 THEN 'p_025_050'
        ELSE 'p_lt_025'
    END AS faixa_p,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao_aval
GROUP BY 1
ORDER BY 1 DESC
''').df())

display(con.execute('''
SELECT
    coalesce(faixa, '(sem)') AS faixa,
    coalesce(regra, '(sem)') AS regra,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao_aval
GROUP BY 1, 2
ORDER BY
    CASE coalesce(faixa, '')
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
        ELSE 9
    END,
    regra
''').df())

display(con.execute('''
SELECT
    n_censos,
    COUNT(*) AS n_cpf,
    n_censos * COUNT(*) AS n_pares
FROM (
    SELECT unique_id_cpf, COUNT(*) AS n_censos
    FROM atribuicao_aval
    GROUP BY 1
)
GROUP BY 1
ORDER BY 1
''').df())


## Qualidade dos pares atribuídos

No par que saiu: mãe, endereço e distância de ano de nascimento.


In [ ]:
display(con.execute(f'''
SELECT
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
    ) / COUNT(*), 1) AS pct_mae_preenchida,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
          AND ca.nome_mae_phon = pb.nome_mae_phon
    ) / COUNT(*), 1) AS pct_mae_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
          AND ca.logradouro_norm = pb.logradouro_norm
    ) / COUNT(*), 1) AS pct_logradouro_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
    ) / COUNT(*), 1) AS pct_cep_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
    ) / COUNT(*), 1) AS pct_data_igual
FROM atribuicao_aval a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
''').df())

display(con.execute(f'''
SELECT
    CASE
        WHEN ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL THEN 'sem_data'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) = 0 THEN 'delta_0'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 1 THEN 'delta_1'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 5 THEN 'delta_2_5'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 10 THEN 'delta_6_10'
        ELSE 'delta_gt_10'
    END AS faixa_delta_ano,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao_aval a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
GROUP BY 1
ORDER BY 1
''').df())


## Quem ficou de fora

Melhor par no parquet por Censo sem atribuição. Quem tinha `p ≥ 0,50` e
mesmo assim não saiu: teto ou falhou o resgate (R1/R2/R3).


In [ ]:
# Mesmo desempate do 04: nota → mãe preenchida → logradouro → CEP.
con.execute(f'''
CREATE OR REPLACE TABLE melhor_qualquer AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY
        p.match_probability DESC,
        CAST(
            ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
            AS INTEGER
        ) DESC,
        CAST(
            ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
            AND ca.logradouro_norm = pb.logradouro_norm
            AS INTEGER
        ) DESC,
        CAST(
            ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
            AS INTEGER
        ) DESC,
        p.unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_piso AS
SELECT *
FROM melhor_qualquer
WHERE match_probability >= {PISO_RESGATE}
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id AS unique_id_censo
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
    SELECT 1 FROM atribuicao_aval a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute(f'''
CREATE OR REPLACE TABLE sem_cpf_melhor AS
SELECT
    s.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    CASE
        WHEN m.match_probability IS NULL THEN 'sem_par'
        WHEN m.match_probability >= 0.50 THEN 'p_ge_050'
        WHEN m.match_probability >= 0.25 THEN 'p_025_050'
        WHEN m.match_probability >= 0.05 THEN 'p_005_025'
        ELSE 'p_lt_005'
    END AS faixa_p
FROM censo_sem_cpf s
LEFT JOIN melhor_qualquer m ON m.unique_id_censo = s.unique_id_censo
''')

n_sem = con.execute('SELECT COUNT(*) FROM censo_sem_cpf').fetchone()[0]
assert n_atrib + n_sem == n_app

display(con.execute(f'''
SELECT
    a.faixa_p,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / {n_sem}, 1) AS pct_sem_cpf,
    ROUND(100.0 * coalesce(g.n, 0) / {n_app}, 1) AS pct_aplicacao
FROM (VALUES
    (1, 'p_ge_050'),
    (2, 'p_025_050'),
    (3, 'p_005_025'),
    (4, 'p_lt_005'),
    (5, 'sem_par')
) a(ord, faixa_p)
LEFT JOIN (
    SELECT faixa_p, COUNT(*) AS n
    FROM sem_cpf_melhor
    GROUP BY 1
) g ON g.faixa_p = a.faixa_p
ORDER BY a.ord
''').df())

# Censos com p >= PISO no melhor-par, mas CPF estourou o teto.
con.execute(f'''
CREATE OR REPLACE TABLE cpf_n_melhor_piso AS
SELECT unique_id_cpf, COUNT(*) AS n_censos
FROM melhor_piso
GROUP BY 1
''')

n_teto = con.execute(f'''
SELECT COUNT(*)
FROM sem_cpf_melhor s
JOIN cpf_n_melhor_piso c ON c.unique_id_cpf = s.unique_id_cpf
WHERE s.match_probability >= {PISO_RESGATE}
  AND c.n_censos > {TETO}
''').fetchone()[0]
n_alto = con.execute(f'''
SELECT COUNT(*) FROM sem_cpf_melhor
WHERE match_probability >= {PISO_RESGATE}
''').fetchone()[0]
print(f'Sem CPF com melhor p >= {PISO_RESGATE}: {n_alto:,}')
print(f'  dos quais teto (CPF com >{TETO} no melhor-par): {n_teto:,}')


## Pares aleatórios por faixa / regra

Amostra nova a cada execução (`ORDER BY random()`). Ajuste `N_AMOSTRA`.


In [ ]:
N_AMOSTRA = 5

cols_par = '''
    a.match_probability,
    a.faixa,
    a.regra,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.nome_mae AS mae_censo,
    pb.nome_mae AS mae_cpf,
    ca.logradouro_norm AS logradouro_censo,
    pb.logradouro_norm AS logradouro_cpf,
    ca.cep AS cep_censo,
    pb.cep AS cep_cpf,
    a.unique_id_censo,
    a.unique_id_cpf
'''

print('--- Por faixa de nota (atribuídos) ---')
for nome, lo, hi in [
    ('p_ge_099', 0.99, 1.01),
    ('p_090_099', 0.90, 0.99),
    ('p_050_090', 0.50, 0.90),
    ('p_025_050', 0.25, 0.50),
]:
    print(f'\n=== {nome} [{lo}, {hi}) ===')
    display(con.execute(f'''
    SELECT {cols_par}
    FROM atribuicao_aval a
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
    WHERE a.match_probability >= {lo}
      AND a.match_probability < {hi}
    ORDER BY random()
    LIMIT {N_AMOSTRA}
    ''').df())

print('\n--- Por regra do 04 ---')
for regra in ['score', 'R1', 'R2', 'R3']:
    print(f'\n=== regra={regra} ===')
    display(con.execute(f'''
    SELECT {cols_par}
    FROM atribuicao_aval a
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
    WHERE a.regra = '{regra}'
    ORDER BY random()
    LIMIT {N_AMOSTRA}
    ''').df())

print('\n--- Sem CPF (melhor par), faixas abaixo do modelo ---')
for nome, lo, hi in [
    ('p_050_090', 0.50, 0.90),
    ('p_025_050', 0.25, 0.50),
    ('p_005_025', 0.05, 0.25),
]:
    print(f'\n=== {nome} [{lo}, {hi}) ===')
    display(con.execute(f'''
    SELECT
        s.match_probability,
        ca.nome_completo AS nome_censo,
        pb.nome_completo AS nome_cpf,
        ca.data_nascimento AS dob_censo,
        pb.data_nascimento AS dob_cpf,
        ca.nome_mae AS mae_censo,
        pb.nome_mae AS mae_cpf,
        ca.logradouro_norm AS logradouro_censo,
        pb.logradouro_norm AS logradouro_cpf,
        s.unique_id_censo,
        s.unique_id_cpf
    FROM sem_cpf_melhor s
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = s.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = s.unique_id_cpf
    WHERE s.match_probability >= {lo}
      AND s.match_probability < {hi}
    ORDER BY random()
    LIMIT {N_AMOSTRA}
    ''').df())


## Observação sobre piso

O 04 já resgata `0,25 ≤ p < 0,90` com R1/R2/R3. Baixar o piso do **modelo**
(`p ≥ 0,90`) sem regra tende a puxar falso positivo — use a amostra acima
antes de afrouxar.


In [ ]:
display(con.execute(f'''
SELECT
    coalesce(faixa, '(sem)') AS faixa,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_atrib,
    ROUND(100.0 * COUNT(*) / {n_app}, 1) AS pct_aplicacao
FROM atribuicao_aval
GROUP BY 1
ORDER BY
    CASE coalesce(faixa, '')
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
        ELSE 9
    END
''').df())


## Lista ouro (censo_limpo)

Universo com a lista ainda dentro. Quem está na ouro e no pareamento conta
como `lista_ouro`. `nos_dois` tem de ser zero (essa gente saiu da aplicação
antes do 04).


In [ ]:
ouro_cols = {
    r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{LISTA_OURO_ARQUIVO}') LIMIT 0"
    ).fetchall()
}
id_ouro, cpf_ouro = lista_ouro_colunas(ouro_cols)

con.execute(f'''
CREATE OR REPLACE TABLE lista_ouro AS
SELECT
    CAST({id_ouro} AS VARCHAR) AS id_censo,
    {cpf_norm_sql(cpf_ouro)} AS cpf_norm
FROM read_parquet('{LISTA_OURO_ARQUIVO}')
WHERE {id_ouro} IS NOT NULL AND {cpf_ouro} IS NOT NULL
''')

display(con.execute(f'''
SELECT
    a.origem,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / nullif(t.n_censo, 0), 1) AS pct
FROM (VALUES
    (1, 'lista_ouro'),
    (2, 'pareamento'),
    (3, 'sem_cpf')
) a(ord, origem)
CROSS JOIN (
    SELECT COUNT(*) AS n_censo FROM {TABELA_CENSO_LIMPA}
) t
LEFT JOIN (
    SELECT
        CASE
            WHEN o.id_censo IS NOT NULL THEN 'lista_ouro'
            WHEN p.unique_id_censo IS NOT NULL THEN 'pareamento'
            ELSE 'sem_cpf'
        END AS origem,
        COUNT(*) AS n
    FROM {TABELA_CENSO_LIMPA} c
    LEFT JOIN (
        SELECT DISTINCT id_censo FROM lista_ouro
    ) o ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
    LEFT JOIN (
        SELECT DISTINCT unique_id_censo FROM atribuicao_aval
    ) p ON p.unique_id_censo = c.unique_id
    GROUP BY 1
) g ON g.origem = a.origem
ORDER BY a.ord
''').df())

n_ambos = con.execute(f'''
SELECT COUNT(*)
FROM {TABELA_CENSO_LIMPA} c
JOIN (SELECT DISTINCT id_censo FROM lista_ouro) o
  ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
JOIN (SELECT DISTINCT unique_id_censo FROM atribuicao_aval) p
  ON p.unique_id_censo = c.unique_id
''').fetchone()[0]
print('nos_dois:', f'{n_ambos:,}')
assert n_ambos == 0


In [8]:
con.close()
print('Fim do 05.')


Fim do 05.
